In [5]:
# ============================================================
# NLLB-200-600M + LoRA (2 GPU, device_map="auto") – Kaggle T4x2
# YÊU CẦU: RESTART KERNEL trước khi chạy
# ============================================================

import os, gc, torch, random
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ---- 1. Cài thư viện + accelerate ----
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q transformers datasets accelerate peft sacrebleu sentencepiece

# Cấu hình accelerate mặc định (không deepspeed, không FSDP)
!accelerate config default 2>/dev/null || true

# ---- 2. Import ----
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, set_seed
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
import sacrebleu

set_seed(42)

# ---- 3. Đường dẫn dữ liệu ----
BASE = "/kaggle/input/datasets/leehoangquan006/iwslt15-envi-data"
TRAIN_SRC = f"{BASE}/train.en"
TRAIN_TGT = f"{BASE}/train.vi"
TEST_SRC  = f"{BASE}/tst2013.en"
TEST_REF  = f"{BASE}/tst2013.vi"

# ---- 4. Đọc & lọc câu (dùng 50k câu, an toàn cho 2 GPU) ----
with open(TRAIN_SRC, encoding="utf-8") as f:
    raw_src = [line.rstrip('\n') for line in f]
with open(TRAIN_TGT, encoding="utf-8") as f:
    raw_tgt = [line.rstrip('\n') for line in f]

pairs = [(s, t) for s, t in zip(raw_src, raw_tgt) if s.strip() and t.strip()]
random.shuffle(pairs)
NUM_TRAIN = 50000
pairs = pairs[:NUM_TRAIN]
src_lines, tgt_lines = zip(*pairs)
src_lines, tgt_lines = list(src_lines), list(tgt_lines)
print(f"✅ Số cặp câu huấn luyện: {len(src_lines):,}")

# ---- 5. Tokenizer & Model (device_map="auto") ----
MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = "eng_Latn"
tokenizer.tgt_lang = "vie_Latn"

# Tự động chia model lên 2 GPU
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)
model.config.tie_word_embeddings = False

# ---- 6. LoRA (vẫn hoạt động trên model phân tán) ----
lora_config = LoraConfig(
    r=8, lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1, bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Bật gradient checkpointing
model.gradient_checkpointing_enable()

# ---- 7. Tokenize (max_length vừa phải) ----
MAX_LENGTH = 64

def preprocess(examples):
    inputs = [tokenizer.src_lang + " " + t for t in examples["en"]]
    targets = [t for t in examples["vi"]]
    return tokenizer(inputs, text_target=targets,
                     max_length=MAX_LENGTH, truncation=True, padding=False)

dataset = Dataset.from_dict({"en": src_lines, "vi": tgt_lines})
tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)

# Giải phóng bộ nhớ CPU
del src_lines, tgt_lines, pairs, raw_src, raw_tgt, dataset
gc.collect()
print(f"🔢 Số mẫu tokenize: {len(tokenized_dataset)}")

# ---- 8. Training Arguments (2 GPU, tăng batch size) ----
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-lora-600M-2gpu",
    per_device_train_batch_size=4,          # mỗi GPU 4 câu
    gradient_accumulation_steps=4,          # effective batch = 4*2*4 = 32
    num_train_epochs=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    save_strategy="no",
    logging_steps=100,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
    predict_with_generate=False,
    push_to_hub=False,
    gradient_checkpointing=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# ---- 9. Huấn luyện ----
print("🚀 Bắt đầu huấn luyện trên 2 GPU...")
trainer.train()
print("✅ Hoàn tất!")

model.save_pretrained("./nllb-lora-600M-2gpu")
tokenizer.save_pretrained("./nllb-lora-600M-2gpu")

# ---- 10. Dịch test (chạy trên 1 GPU, vì chỉ có 1 model) ----
with open(TEST_SRC, encoding="utf-8") as f:
    test_lines = [l.strip() for l in f if l.strip()]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # đưa toàn bộ về GPU0 để dịch
model.eval()

BATCH_SIZE = 8
translations = []
vie_id = tokenizer.convert_tokens_to_ids("vie_Latn")

for i in range(0, len(test_lines), BATCH_SIZE):
    batch = test_lines[i:i+BATCH_SIZE]
    inputs = [tokenizer.src_lang + " " + t for t in batch]
    enc = tokenizer(inputs, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        out = model.generate(**enc, forced_bos_token_id=vie_id,
                             num_beams=5, max_new_tokens=MAX_LENGTH,
                             early_stopping=True)
    translations += tokenizer.batch_decode(out, skip_special_tokens=True)

# ---- 11. Lưu results.csv ----
with open("results.csv", "w", encoding="utf-8") as f:
    f.write("Vietnamese\n")
    for t in translations:
        escaped = t.replace('"', '""')
        f.write(f'"{escaped}"\n')
print("💾 results.csv đã sẵn sàng")

# ---- 12. BLEU ----
if os.path.exists(TEST_REF):
    with open(TEST_REF, encoding="utf-8") as f:
        refs = [l.strip() for l in f if l.strip()]
    if len(translations) == len(refs):
        bleu = sacrebleu.corpus_bleu(translations, [refs])
        print(f"🎯 BLEU trên tst2013: {bleu.score:.2f}")
    else:
        print("⚠️ Số dòng không khớp, bỏ qua BLEU.")
else:
    print("ℹ️ Không có tst2013.vi")

print("🏁 HOÀN THÀNH")

accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml
✅ Số cặp câu huấn luyện: 50,000


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 1,179,648 || all params: 1,403,318,272 || trainable%: 0.0841


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

🔢 Số mẫu tokenize: 50000
🚀 Bắt đầu huấn luyện trên 2 GPU...


Step,Training Loss
100,10.052953
200,7.328542
300,7.084417
400,6.886708
500,6.913097
600,6.747370
700,6.934550
800,6.842542
900,6.799748
1000,6.562914


✅ Hoàn tất!


RuntimeError: Expected all tensors to be on the same device, but got weight is on cuda:0, different from other tensors on cuda:1 (when checking argument in method wrapper_CUDA__native_layer_norm)

In [8]:
# ============================================================
# CHỈ DỊCH TỪ MODEL ĐÃ HUẤN LUYỆN (không train lại)
# ============================================================
import os, gc, torch
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!pip uninstall -y torchao 2>/dev/null || true
!pip install -q transformers accelerate peft sacrebleu sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import sacrebleu

# ---- Đường dẫn ----
BASE = "/kaggle/input/datasets/leehoangquan006/iwslt15-envi-data"
TEST_SRC  = f"{BASE}/tst2013.en"
TEST_REF  = f"{BASE}/tst2013.vi"
ADAPTER_DIR = "./nllb-lora-600M-2gpu"   # nơi lưu adapter đã train
MODEL_NAME = "facebook/nllb-200-distilled-600M"
MAX_LENGTH = 64

# ---- Tải tokenizer và base model (CPU, không device_map) ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = "eng_Latn"
tokenizer.tgt_lang = "vie_Latn"

print("🔄 Đang tải base model (CPU)...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
base_model.config.tie_word_embeddings = False

# ---- Gắn adapter đã lưu và merge ----
print("🔗 Đang nạp adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()   # gộp adapter, trả model sạch

# Đưa lên GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print("✅ Model sẵn sàng.")

# ---- Đọc file test ----
with open(TEST_SRC, encoding="utf-8") as f:
    test_lines = [l.strip() for l in f if l.strip()]

BATCH_SIZE = 8
translations = []
vie_id = tokenizer.convert_tokens_to_ids("vie_Latn")

# ---- Dịch ----
for i in range(0, len(test_lines), BATCH_SIZE):
    batch = test_lines[i:i+BATCH_SIZE]
    inputs = [tokenizer.src_lang + " " + t for t in batch]
    enc = tokenizer(inputs, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        out = model.generate(**enc, forced_bos_token_id=vie_id,
                             num_beams=5, max_new_tokens=MAX_LENGTH,
                             early_stopping=True)
    translations += tokenizer.batch_decode(out, skip_special_tokens=True)

# ---- Lưu results.csv ----
with open("results.csv", "w", encoding="utf-8") as f:
    f.write("Vietnamese\n")
    for t in translations:
        escaped = t.replace('"', '""')
        f.write(f'"{escaped}"\n')
print("💾 results.csv đã sẵn sàng")

# ---- BLEU ----
if os.path.exists(TEST_REF):
    with open(TEST_REF, encoding="utf-8") as f:
        refs = [l.strip() for l in f if l.strip()]
    if len(translations) == len(refs):
        bleu = sacrebleu.corpus_bleu(translations, [refs])
        print(f"🎯 BLEU trên tst2013: {bleu.score:.2f}")
    else:
        print("⚠️ Số dòng không khớp, bỏ qua BLEU.")
else:
    print("ℹ️ Không có tst2013.vi")

print("🏁 HOÀN THÀNH! Nộp results.csv lên Kaggle.")

🔄 Đang tải base model (CPU)...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🔗 Đang nạp adapter...
✅ Model sẵn sàng.
💾 results.csv đã sẵn sàng


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


🎯 BLEU trên tst2013: 33.88
🏁 HOÀN THÀNH! Nộp results.csv lên Kaggle.
